In [2]:
import os  
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')  
if gpus:  
    tf.config.experimental.set_memory_growth(gpus[0], True)
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

### Zadanie 3

In [4]:
kernel = 3
leng = 32
channels = 3
filters = 32

conv2d = keras.Sequential([
    keras.Input(shape=(leng, leng, channels)),
    keras.layers.Conv2D(filters, kernel, padding='same', activation="relu")
])

conv2d_sep = keras.Sequential([
    keras.Input(shape=(leng, leng, channels)),
    keras.layers.SeparableConv2D(filters, kernel, padding='same', activation="relu")
])

params_2d = (kernel * kernel * channels + 1) * leng
params_sep = (kernel * kernel * 1 + 1) * channels
params_sep = (1 * 1 * channels + 1) * filters + params_sep

print(params_2d)
print(params_sep)

print(f'Liczba parametrów conv2d: {conv2d.count_params()}')
print(f'Liczba parametrów separable conv2d: {conv2d_sep.count_params()}')


896
158
Liczba parametrów conv2d: 896
Liczba parametrów separable conv2d: 155


### Zadanie 4

In [5]:
conv_regular = keras.Sequential([
    keras.Input(shape=(14, 14, 128)),
    keras.layers.Conv2D(256, (3,3), padding='same', activation="relu")
])
conv_bottleneck = keras.Sequential([
    keras.Input(shape=(14, 14, 128)),
    keras.layers.Conv2D(32, (1,1), padding='same', activation="relu"),
    keras.layers.Conv2D(32, (3,3), padding='same', activation="relu"),
    keras.layers.Conv2D(256, (1,1), padding='same', activation="relu")
])

print(f'Liczba parametrów conv2d: {conv_regular.count_params()}')
print(f'Liczba parametrów bottleneck conv2d: {conv_bottleneck.count_params()}')

Liczba parametrów conv2d: 295168
Liczba parametrów bottleneck conv2d: 21824


### Zadanie 7

In [7]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train[0].shape

dim = X_train.max()
X_train = X_train.astype('float32') / dim
X_test = X_test.astype('float32') / dim
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]
num_classes = len(np.unique(y_train))
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)
models = {'Dropout': keras.Sequential([
            keras.Input(shape=(28,28,1)),
            layers.Conv2D(32, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.Conv2D(32, kernel_size=(3,3), padding='same', activation='relu'),
            layers.MaxPooling2D(pool_size=(2,2)),
            layers.Dropout(0.25),
            layers.Conv2D(64, kernel_size=(3,3), padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.GlobalAveragePooling2D(),
            layers.Dense(64, activation='softmax'),
            layers.Dropout(0.25),
            layers.Dense(num_classes, activation='softmax')
    ]),
          'Spatial_Dropout': keras.Sequential([
            keras.Input(shape=(28,28,1)),
            layers.Conv2D(32, 3, padding='same', use_bias=False),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Conv2D(32, 3, padding='same', use_bias=False),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.MaxPooling2D(2),
            layers.SpatialDropout2D(0.25),
            layers.Conv2D(64, 3, padding='same', use_bias=False),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Conv2D(64, 3, padding='same', use_bias=False),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.MaxPooling2D(2),
            layers.SpatialDropout2D(0.25),
            layers.GlobalAveragePooling2D(),
            layers.Dense(num_classes, activation='softmax')              
])}

results = {}
for k, v in models.items():
    v.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    results[k] = v.fit(X_train,
                      y_train,
                      batch_size=128,
                      epochs=15,
                      validation_data=(X_val, y_val),
                      	callbacks=[
                    		EarlyStopping(patience=4, restore_best_weights=True),
                    		ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)
                    		],
                      verbose=0)

In [8]:
df = pd.DataFrame({
    name: history.history['val_accuracy']
    for name, history in results.items()
})
df

,Dropout,Spatial_Dropout
0,0.100333,0.260500
1,0.543833,0.835167
2,0.548000,0.872333
3,0.582333,0.867000
4,0.566500,0.884333
5,0.596000,0.880000
6,0.611167,0.867833
7,0.720000,0.904833
8,0.800500,0.908667
9,0.841333,0.908167
